In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("computingvictor/transactions-fraud-datasets")

print("Path to dataset files:", path)

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 348M/348M [00:07<00:00, 47.4MB/s] 

Extracting files...


Path to dataset files: /home/codespace/.cache/kagglehub/datasets/computingvictor/transactions-fraud-datasets/versions/1


In [11]:
import pandas as pd

transactions_df=pd.read_csv("/home/codespace/.cache/kagglehub/datasets/computingvictor/transactions-fraud-datasets/versions/1/transactions_data.csv")
transactions_df.head(5)

: 

In [7]:
users_df=pd.read_csv("/home/codespace/.cache/kagglehub/datasets/computingvictor/transactions-fraud-datasets/versions/1/users_data.csv")
users_df.head(5)

,id,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
0,825,53,66,1966,11,Female,462 Rose Lane,34.15,-117.76,$29278,$59696,$127613,787,5
1,1746,53,68,1966,12,Female,3606 Federal Boulevard,40.76,-73.74,$37891,$77254,$191349,701,5
2,1718,81,67,1938,11,Female,766 Third Drive,34.02,-117.89,$22681,$33483,$196,698,5
3,708,63,63,1957,1,Female,3 Madison Street,40.71,-73.99,$163145,$249925,$202328,722,4
4,1164,43,70,1976,9,Male,9620 Valley Stream Drive,37.76,-122.44,$53797,$109687,$183855,675,1


In [8]:
cards_df=pd.read_csv("/home/codespace/.cache/kagglehub/datasets/computingvictor/transactions-fraud-datasets/versions/1/cards_data.csv")
cards_df.head(5)

,id,client_id,card_brand,card_type,card_number,expires,cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web
0,4524,825,Visa,Debit,4344676511950444,12/2022,623,YES,2,$24295,09/2002,2008,No
1,2731,825,Visa,Debit,4956965974959986,12/2020,393,YES,2,$21968,04/2014,2014,No
2,3701,825,Visa,Debit,4582313478255491,02/2024,719,YES,2,$46414,07/2003,2004,No
3,42,825,Visa,Credit,4879494103069057,08/2024,693,NO,1,$12400,01/2003,2012,No
4,4659,825,Mastercard,Debit (Prepaid),5722874738736011,03/2009,75,YES,1,$28,09/2008,2009,No


In [15]:
import pandas as pd
import matplotlib.pyplot as plt
import json

# 1. Correctly open and load the JSON file's data
file_path = "/home/codespace/.cache/kagglehub/datasets/computingvictor/transactions-fraud-datasets/versions/1/mcc_codes.json"
with open(file_path, 'r') as f:
    mcc_data = json.load(f)

# 2. Convert key-value pairs to rows and name your columns
mcc_df = pd.DataFrame(list(mcc_data.items()), columns=['mcc_code', 'description'])

# 3. Set your newly named business description or code as the index
mcc_df.set_index('mcc_code', inplace=True)

# Print clean structural output
print(df.head())

                                   description
mcc_code                                      
5812             Eating Places and Restaurants
5541                          Service Stations
7996      Amusement Parks, Carnivals, Circuses
5411              Grocery Stores, Supermarkets
4784                     Tolls and Bridge Fees


In [6]:
import json
import sqlite3
import pandas as pd

# 1. Connect to the database file
conn = sqlite3.connect('fraud_data.db')

# --- PART A: PROCESS MCC CODES (Small JSON file - safe to read all at once) ---
json_path = "/home/codespace/.cache/kagglehub/datasets/computingvictor/transactions-fraud-datasets/versions/1/mcc_codes.json"
with open(json_path, 'r') as f:
    mcc_data = json.load(f)

mcc_df = pd.DataFrame(list(mcc_data.items()), columns=['mcc_code', 'description'])
mcc_df.to_sql('mcc', conn, if_exists='replace', index=False)
print("✔ MCC codes table written.")


# --- PART B: PROCESS CARDS DATA (Medium CSV file - read in chunks) ---
cards_path = "/home/codespace/.cache/kagglehub/datasets/computingvictor/transactions-fraud-datasets/versions/1/cards_data.csv"
chunk_size = 50000

# Write the first chunk with 'replace' to clear out old data, then append the rest
first_chunk = True
for chunk in pd.read_csv(cards_path, chunksize=chunk_size):
    if first_chunk:
        chunk.to_sql('cards', conn, if_exists='replace', index=False)
        first_chunk = False
    else:
        chunk.to_sql('cards', conn, if_exists='append', index=False)
print("✔ Cards table written in chunks.")


# --- PART C: PROCESS TRANSACTIONS (Massive CSV file - read in chunks) ---
csv_path = "/home/codespace/.cache/kagglehub/datasets/computingvictor/transactions-fraud-datasets/versions/1/transactions_data.csv"

# Write the first chunk with 'replace' to clear out old data, then append the rest
first_chunk = True
for chunk in pd.read_csv(csv_path, chunksize=chunk_size):
    if first_chunk:
        chunk.to_sql('transactions', conn, if_exists='replace', index=False)
        first_chunk = False
    else:
        chunk.to_sql('transactions', conn, if_exists='append', index=False)
print("✔ Transactions table written in chunks.")

csv_path = "/home/codespace/.cache/kagglehub/datasets/computingvictor/transactions-fraud-datasets/versions/1/users_data.csv"
first_chunk = True
for chunk in pd.read_csv(csv_path, chunksize=chunk_size):
    if first_chunk:
        chunk.to_sql('users', conn, if_exists='replace', index=False)
        first_chunk = False
    else:
        chunk.to_sql('users', conn, if_exists='append', index=False)
print("✔ Users table written in chunks.")


# --- PART D: CLEAN UP ---
print(" All tables successfully written to fraud_data.db without crashing!")
conn.close()


✔ MCC codes table written.
✔ Cards table written in chunks.
✔ Transactions table written in chunks.
✔ Users table written in chunks.
 All tables successfully written to fraud_data.db without crashing!


In [6]:
import sqlite3
import pandas as pd

# Open the database file
conn = sqlite3.connect('fraud_data.db')

# SQL query
sql_query = """
SELECT 
t.client_id,t.date,t.card_id,t.amount,merchant_id,t.zip,t.mcc,t.errors,
m.description,
u.current_age,u.retirement_age,u.gender,u.yearly_income,u.total_debt,u.num_credit_cards

FROM transactions t
INNER JOIN mcc m ON m.mcc_code =  t.mcc
INNER JOIN users u ON t.client_id = u.id
INNER JOIN cards c ON t.card_id = c.id
LIMIT 5;
"""

# Extract directly into a fresh DataFrame
joined_df = pd.read_sql_query(sql_query, conn)
conn.close()

print(joined_df)

   client_id                 date  card_id  amount  merchant_id      zip  \
0       1963  2010-01-01 01:07:00     3364  $48.12        88646  95688.0   
1       1642  2010-01-01 01:11:00     4281   $3.96        44625   8736.0   
2        247  2010-01-01 01:41:00     4155   $3.29        92883  70605.0   
3       1642  2010-01-01 01:56:00     4281   $3.35        44625   8736.0   
4        474  2010-01-01 02:14:00     2493  $91.02        35451  11596.0   

    mcc errors                    description  current_age  retirement_age  \
0  5812   None  Eating Places and Restaurants           98              69   
1  5812   None  Eating Places and Restaurants           37              66   
2  5812   None  Eating Places and Restaurants           49              69   
3  5812   None  Eating Places and Restaurants           37              66   
4  5812   None  Eating Places and Restaurants           55              66   

   gender yearly_income total_debt  num_credit_cards  
0    Male        $3